<a href="https://colab.research.google.com/github/quasar-in-the-mist/Econo-Forecasting/blob/main/Non_Gaussianity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Constraints and pseudo-random variables
# Simulating some noisy data from a system with many (m) degrees of
# freedom but with hidden (n >3) n-point correlations within those degrees
# of freedom that introduces subtle-non-gaussianity, then generate
# independent tests of non-gaussianity that can maybe detect these
# deviations from non-gaussianity, it could be some scalar functions on a
# lattice of points on a sphere surface as the data format to make and digest.

# Non-Gaussianity Injection (the hard part)
# The injections are designed to be degenerate with Gaussianity at 2-point level:

# Direct n-point coupling: For a group {i,j,k,l}, we add ε · x_j·x_k·x_l / (1 + Σx²) to x_i. The denominator ensures the perturbation is bounded and the normalization suppresses large deviations — making the effect subtle. The connected 4-point function ⟨x_i x_j x_k x_l⟩_c becomes non-zero while 2-point functions are barely affected.

# Copula manipulation: Transform to uniform marginals, perturb in quantile space using products of Hermite polynomials H₂(z_i)·H₂(z_j)·..., then transform back. This directly injects non-Gaussian copula structure while preserving marginals exactly.

In [ ]:
import numpy as np
from scipy import stats, special, spatial
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ============================================================
# PART 1: SPHERICAL LATTICE CONSTRUCTION
# ============================================================

def fibonacci_sphere(num_points):
    """Generate approximately uniform points on a sphere via Fibonacci spiral."""
    indices = np.arange(0, num_points, dtype=float) + 0.5
    phi = np.arccos(1 - 2 * indices / num_points)  # polar angle
    theta = np.pi * (1 + 5**0.5) * indices           # azimuthal angle
    x = np.sin(phi) * np.cos(theta)
    y = np.sin(phi) * np.sin(theta)
    z = np.cos(phi)
    return np.column_stack([x, y, z]), theta % (2*np.pi), phi

def angular_distance_matrix(coords):
    """Compute geodesic distance matrix on unit sphere."""
    # Using dot product -> arccos for angular separation
    dot = np.clip(coords @ coords.T, -1, 1)
    return np.arccos(dot)

m = 512  # degrees of freedom (points on sphere)
coords, theta, phi = fibonacci_sphere(m)
ang_dist = angular_distance_matrix(coords)

print(f"Sphere lattice: {m} points")
print(f"Mean nearest-neighbor angular separation: "
      f"{np.sort(ang_dist, axis=1)[:,1].mean():.4f} rad "
      f"({np.degrees(np.sort(ang_dist, axis=1)[:,1].mean()):.2f} deg)")

# ============================================================
# PART 2: SPATIAL CORRELATION STRUCTURE (GAUSSIAN BASELINE)
# ============================================================

def make_gaussian_covariance(ang_dist, ell_max=20, C_ell=None):
    """
    Build covariance matrix from angular power spectrum C_ell
    using Legendre polynomial expansion:
        C(theta) = sum_ell (2ell+1)/(4pi) * C_ell * P_ell(cos(theta))
    """
    cos_dist = np.cos(ang_dist)
    if C_ell is None:
        # Realistic-ish power spectrum: power law with cutoff
        ells = np.arange(ell_max + 1)
        C_ell = np.zeros(ell_max + 1)
        C_ell[1:] = 1.0 / (ells[1:] * (ells[1:] + 1))  # ~1/ell^2 scaling
        C_ell[0] = 0  # no monopole

    cov = np.zeros_like(ang_dist)
    for ell in range(len(C_ell)):
        P_ell = special.eval_legendre(ell, cos_dist)
        cov += (2*ell + 1) / (4*np.pi) * C_ell[ell] * P_ell

    # Regularize for numerical stability
    cov += 1e-8 * np.eye(len(cov))
    return cov, C_ell

cov_matrix, C_ell = make_gaussian_covariance(ang_dist, ell_max=25)

# Cholesky decomposition for sampling
L_chol = np.linalg.cholesky(cov_matrix)

print(f"Covariance matrix condition number: {np.linalg.cond(cov_matrix):.1f}")
print(f"Power spectrum ell_max = {len(C_ell)-1}")

# ============================================================
# PART 3: HIDDEN N-POINT CORRELATION INJECTION
# ============================================================

class HiddenCorrelationInjector:
    """
    Injects subtle n-point correlations (n=4,5,6) that:
    1. Preserve marginal distributions (approximately Gaussian at each point)
    2. Preserve 2-point correlation structure
    3. Introduce higher-order connected correlations among spatial neighbors

    Strategy: Use a carefully calibrated nonlinear transformation on
    spatially-coupled groups of points, then re-Gaussianize marginals
    while preserving the higher-order coupling signature.
    """

    def __init__(self, coords, ang_dist, n_groups=30, group_orders=[4, 5, 6],
                 coupling_strength=0.15):
        self.coords = coords
        self.ang_dist = ang_dist
        self.m = len(coords)
        self.n_groups = n_groups
        self.group_orders = group_orders
        self.coupling_strength = coupling_strength
        self.groups = self._build_correlation_groups()

    def _build_correlation_groups(self):
        """
        Build groups of spatially nearby points that will share
        hidden n-point correlations. Each group has n members where
        n is drawn from group_orders.
        """
        groups = []
        rng = np.random.RandomState(123)

        for _ in range(self.n_groups):
            n = rng.choice(self.group_orders)
            # Pick a seed point, then find its nearest neighbors
            seed = rng.randint(self.m)
            neighbors = np.argsort(self.ang_dist[seed])[:n]
            groups.append(neighbors)

        return groups

    def inject(self, gaussian_field, rng=None):
        """
        Inject hidden n-point correlations into a Gaussian random field.

        Method: For each group of n points, introduce a shared latent
        variable that creates connected n-point functions without
        significantly affecting the 2-point function.

        Key technique: "Edgeworth-like" perturbation
        f(x1,...,xn) -> f(x1,...,xn) * [1 + epsilon * h_n(x1,...,xn)]
        implemented via acceptance-rejection-like reweighting in sample space.
        """
        if rng is None:
            rng = np.random.RandomState()

        field = gaussian_field.copy()
        eps = self.coupling_strength

        for group in self.groups:
            n = len(group)
            vals = field[group]

            if n == 4:
                # 4-point connected correlation: kurtosis-like coupling
                # Introduce: <x1 x2 x3 x4>_c != 0
                # Via shared latent variable: x_i -> x_i + eps * (prod_{j!=i} x_j) / (1 + sum x_j^2)
                product_others = np.array([
                    np.prod(vals[np.arange(n) != i]) for i in range(n)
                ])
                norm = 1 + np.sum(vals**2)
                perturbation = eps * product_others / (norm * np.sqrt(n))
                field[group] += perturbation

            elif n == 5:
                # 5-point: odd correlator via sinh-like coupling
                # <x1 x2 x3 x4 x5>_c != 0
                product_others = np.array([
                    np.prod(vals[np.arange(n) != i]) for i in range(n)
                ])
                norm = 1 + np.sum(vals**4)
                perturbation = eps * np.tanh(product_others) / (norm**(1/4) * n)
                field[group] += perturbation

            elif n == 6:
                # 6-point: use triad coupling — pairs of triplets
                # Split into two triplets and couple their products
                half = n // 2
                prod1 = np.prod(vals[:half])
                prod2 = np.prod(vals[half:])

                for i in range(half):
                    others1 = np.prod(vals[:half][np.arange(half) != i])
                    field[group[i]] += eps * others1 * prod2 / (1 + prod1**2 + prod2**2)
                for i in range(half, n):
                    j = i - half
                    others2 = np.prod(vals[half:][np.arange(half) != j])
                    field[group[i]] += eps * others2 * prod1 / (1 + prod1**2 + prod2**2)

        return field

    def copula_inject(self, gaussian_field, rng=None):
        """
        Alternative injection via copula manipulation:
        Transform to uniform marginals, apply a non-Gaussian copula
        perturbation on groups, transform back.
        """
        if rng is None:
            rng = np.random.RandomState()

        field = gaussian_field.copy()
        eps = self.coupling_strength

        # Transform each point to its quantile (using empirical or theoretical CDF)
        # Here we use the theoretical Gaussian CDF since the field is Gaussian
        sigma = np.sqrt(np.diag(cov_matrix))
        u = stats.norm.cdf(field / sigma)  # uniform marginals

        for group in self.groups:
            n = len(group)
            u_g = u[group]

            # Map to Gaussian quantiles
            z_g = stats.norm.ppf(np.clip(u_g, 1e-8, 1-1e-8))

            # Apply Hermite polynomial coupling
            if n >= 4:
                # H_2(x) = x^2 - 1 (Hermite)
                h2 = z_g**2 - 1
                # Coupled perturbation using product of Hermite polynomials
                for i in range(n):
                    others_h2 = np.prod(h2[np.arange(n) != i])
                    # Small perturbation to the quantile
                    z_g[i] += eps * others_h2 / (n * (1 + np.sum(h2**2)))

            # Map back through CDF
            u_new = stats.norm.cdf(z_g)
            field[group] = sigma[group] * stats.norm.ppf(np.clip(u_new, 1e-8, 1-1e-8))

        return field

injector = HiddenCorrelationInjector(
    coords, ang_dist,
    n_groups=40,          # number of correlation groups
    group_orders=[4, 5, 6],  # n-point orders
    coupling_strength=0.12   # subtle!
)

# ============================================================
# PART 4: GENERATE ENSEMBLE OF REALIZATIONS
# ============================================================

n_realizations = 500

def generate_gaussian_field(L_chol, rng):
    """Generate one Gaussian random field realization on the sphere."""
    z = rng.randn(L_chol.shape[0])
    return L_chol @ z

def generate_nongaussian_field(L_chol, injector, rng):
    """Generate a field with hidden non-Gaussianity."""
    g_field = generate_gaussian_field(L_chol, rng)
    # Apply both injection methods for richer correlation structure
    ng_field = injector.inject(g_field, rng)
    ng_field = injector.copula_inject(ng_field, rng)
    return ng_field

print("\nGenerating ensembles...")
rng = np.random.RandomState(0)

gaussian_ensemble = np.array([
    generate_gaussian_field(L_chol, rng) for _ in range(n_realizations)
])

rng2 = np.random.RandomState(0)  # same seeds for fair comparison
nongaussian_ensemble = np.array([
    generate_nongaussian_field(L_chol, injector, rng2) for _ in range(n_realizations)
])

print(f"Gaussian ensemble shape: {gaussian_ensemble.shape}")
print(f"Non-Gaussian ensemble shape: {nongaussian_ensemble.shape}")

# Quick sanity checks
print(f"\n--- Sanity Checks (should be similar) ---")
print(f"Gaussian mean of means: {gaussian_ensemble.mean(axis=1).mean():.6f}")
print(f"NonGauss mean of means: {nongaussian_ensemble.mean(axis=1).mean():.6f}")
print(f"Gaussian mean of stds:  {gaussian_ensemble.std(axis=1).mean():.6f}")
print(f"NonGauss mean of stds:  {nongaussian_ensemble.std(axis=1).mean():.6f}")

# ============================================================
# PART 5: NON-GAUSSIANITY TESTS
# ============================================================

class NonGaussianityTestSuite:
    """
    Suite of independent tests for non-Gaussianity, each targeting
    different statistical signatures.
    """

    def __init__(self, coords, ang_dist, cov_matrix):
        self.coords = coords
        self.ang_dist = ang_dist
        self.m = len(coords)
        self.cov_matrix = cov_matrix
        self.cov_inv = np.linalg.inv(cov_matrix)
        self.L_inv = np.linalg.inv(np.linalg.cholesky(cov_matrix))

    # ----- TEST 1: Minkowski Functionals -----
    def minkowski_functionals(self, field, thresholds=None):
        """
        Compute Minkowski functionals (area, perimeter, genus) of
        excursion sets at multiple thresholds.

        For a scalar field on a sphere, the excursion set at threshold nu
        is the region where f > nu*sigma. The three Minkowski functionals
        capture morphological information sensitive to non-Gaussianity.
        """
        if thresholds is None:
            thresholds = np.linspace(-2.5, 2.5, 20)

        sigma = np.std(field)
        normalized = field / sigma

        # Build adjacency from Delaunay-like triangulation on sphere
        # Use convex hull of 3D points (= Delaunay on sphere)
        hull = spatial.ConvexHull(self.coords)

        V0 = []  # Area fraction (fraction above threshold)
        V1 = []  # Boundary length proxy
        V2 = []  # Euler characteristic / genus

        for nu in thresholds:
            excursion = normalized > nu

            # V0: fraction of points above threshold
            v0 = np.mean(excursion)

            # V1: fraction of edges crossing the threshold
            n_boundary = 0
            n_edges = 0
            for simplex in hull.simplices:
                for i, j in combinations(simplex, 2):
                    n_edges += 1
                    if excursion[i] != excursion[j]:
                        n_boundary += 1
            v1 = n_boundary / max(n_edges, 1)

            # V2: Euler characteristic via vertices - edges + faces in excursion
            # Simplified: count connected "hot" and "cold" regions
            # Use graph-based approach
            v2 = self._euler_characteristic(excursion, hull.simplices)

            V0.append(v0)
            V1.append(v1)
            V2.append(v2)

        return np.array(V0), np.array(V1), np.array(V2), thresholds

    def _euler_characteristic(self, excursion, simplices):
        """Compute Euler characteristic of excursion set on triangulated sphere."""
        # Count vertices, edges, faces in the excursion set
        vertices = set(np.where(excursion)[0])
        edges = set()
        faces = 0

        for simplex in simplices:
            in_exc = [v in vertices for v in simplex]
            # Edges
            for i, j in combinations(range(3), 2):
                if in_exc[i] and in_exc[j]:
                    edge = tuple(sorted([simplex[i], simplex[j]]))
                    edges.add(edge)
            # Faces
            if all(in_exc):
                faces += 1

        V = len(vertices)
        E = len(edges)
        F = faces
        return V - E + F

    def minkowski_test_statistic(self, field, thresholds=None):
        """Combine Minkowski functionals into a single test statistic."""
        V0, V1, V2, th = self.minkowski_functionals(field, thresholds)
        # Deviation from Gaussian expectation for V0
        # For Gaussian: V0(nu) = 1 - Phi(nu)
        sigma = np.std(field)
        expected_V0 = 1 - stats.norm.cdf(th)

        # Chi-squared-like statistic for V0 deviation
        residual_V0 = V0 - expected_V0
        stat_V0 = np.sum(residual_V0**2) / len(th)

        # For V2: Gaussian expectation is (2pi)^(-1) * nu * exp(-nu^2/2) * A/(4pi)
        # Simplified: just use the raw sum of squared V2 as a feature
        stat_V2 = np.sum(np.array(V2)**2) / len(th)

        return stat_V0 + 0.1 * stat_V2

    # ----- TEST 2: Whitened Cumulants -----
    def whitened_cumulants(self, field):
        """
        Whiten the field using the known covariance, then compute
        excess kurtosis and higher cumulants of the whitened components.

        If the field is truly Gaussian, whitened components are iid N(0,1),
        so any departure in higher moments signals non-Gaussianity.
        """
        # Whiten: w = L^{-1} @ field
        w = self.L_inv @ field

        # Individual whitened component statistics
        k3 = stats.moment(w, moment=3)  # should be 0
        k4 = stats.kurtosis(w, fisher=True)  # excess kurtosis, should be 0
        k5 = stats.moment(w, moment=5)  # should be 0
        k6 = stats.moment(w, moment=6) - 15  # 6th moment of N(0,1) is 15

        return k3, k4, k5, k6

    def whitened_cumulant_test(self, field):
        """Test statistic from whitened cumulants."""
        k3, k4, k5, k6 = self.whitened_cumulants(field)
        # Combine into chi-squared-like statistic
        # Variance of k4 for m samples of N(0,1) ~ 24/m
        # Variance of k3 ~ 6/m
        var_k3 = 6.0 / self.m
        var_k4 = 24.0 / self.m
        var_k5 = 720.0 / self.m  # approximate
        var_k6 = 100.0  # rough estimate

        stat = k3**2/var_k3 + k4**2/var_k4 + k5**2/var_k5 + k6**2/var_k6
        return stat

    # ----- TEST 3: Bispectrum Estimator -----
    def bispectrum_binned(self, field, n_bins=8):
        """
        Estimate the angular bispectrum in configuration space.

        The bispectrum B(theta1, theta2, theta3) is the 3-point function
        averaged over triangles with given angular separations.
        Non-zero connected bispectrum -> non-Gaussianity.
        """
        sigma = np.std(field)
        f = field / sigma  # normalize

        bin_edges = np.linspace(0, np.pi, n_bins + 1)
        bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

        # Sample random triangles for efficiency
        rng = np.random.RandomState(777)
        n_triangles = 5000

        bispectrum = np.zeros((n_bins, n_bins, n_bins))
        counts = np.zeros((n_bins, n_bins, n_bins))

        for _ in range(n_triangles):
            i, j, k = rng.choice(self.m, 3, replace=False)
            d_ij = self.ang_dist[i, j]
            d_jk = self.ang_dist[j, k]
            d_ik = self.ang_dist[i, k]

            b_ij = np.searchsorted(bin_edges[1:], d_ij)
            b_jk = np.searchsorted(bin_edges[1:], d_jk)
            b_ik = np.searchsorted(bin_edges[1:], d_ik)

            if b_ij < n_bins and b_jk < n_bins and b_ik < n_bins:
                # Sort bin indices for symmetry
                bins = tuple(sorted([b_ij, b_jk, b_ik]))
                bispectrum[bins] += f[i] * f[j] * f[k]
                counts[bins] += 1

        # Normalize
        mask = counts > 0
        bispectrum[mask] /= counts[mask]

        return bispectrum, counts, bin_centers

    def bispectrum_test(self, field):
        """Test statistic from binned bispectrum."""
        bispec, counts, _ = self.bispectrum_binned(field)
        mask = counts > 10  # only bins with enough samples
        if mask.sum() == 0:
            return 0.0
        # Sum of squared bispectrum values (should be ~0 for Gaussian)
        return np.sum(bispec[mask]**2 * counts[mask]) / mask.sum()

    # ----- TEST 4: Phase Correlation Test -----
    def phase_correlations(self, field):
        """
        Decompose field into spherical harmonic-like components using
        the covariance eigenbasis, then test for phase correlations.

        For Gaussian fields, Fourier/harmonic phases are uniformly
        distributed and independent. Non-Gaussianity introduces
        phase correlations.
        """
        # Use eigenbasis of covariance as harmonic decomposition
        eigenvalues, eigenvectors = np.linalg.eigh(self.cov_matrix)

        # Project field onto eigenmodes
        coeffs = eigenvectors.T @ field

        # Normalize by expected amplitude
        amplitudes = np.abs(coeffs) / np.sqrt(np.maximum(eigenvalues, 1e-10))
        phases = np.angle(coeffs + 1j * np.roll(coeffs, 1))  # pseudo-phase

        # Test 1: Uniformity of phases (Rayleigh test)
        # Compute mean resultant length for groups of phases
        n_groups = 10
        group_size = len(phases) // n_groups
        phase_stats = []
        for g in range(n_groups):
            ph = phases[g*group_size:(g+1)*group_size]
            R = np.abs(np.mean(np.exp(1j * ph)))
            phase_stats.append(R)

        # Test 2: Cross-phase correlations
        n_pairs = min(200, len(phases)*(len(phases)-1)//2)
        rng = np.random.RandomState(555)
        cross_corr = []
        for _ in range(n_pairs):
            i, j = rng.choice(len(phases), 2, replace=False)
            cross_corr.append(np.cos(phases[i] - phases[j]))

        mean_cross = np.mean(cross_corr)
        mean_resultant = np.mean(phase_stats)

        return mean_resultant**2 * n_groups + mean_cross**2 * n_pairs

    # ----- TEST 5: Nearest-Neighbor Joint Distribution Test -----
    def nn_joint_test(self, field, k_neighbors=5):
        """
        Test the joint distribution of field values at neighboring points.

        For Gaussian fields, the joint distribution of (f_i, f_j) for
        neighbors i,j is bivariate Gaussian. Test for departures using
        a copula-based approach.
        """
        sigma = np.sqrt(np.diag(self.cov_matrix))

        # For each point, get k nearest neighbors
        nn_indices = np.argsort(self.ang_dist, axis=1)[:, 1:k_neighbors+1]

        # Collect (u_i, u_j) pairs where u = Phi(f/sigma)
        u_i_list = []
        u_j_list = []

        for i in range(self.m):
            for j in nn_indices[i]:
                u_i = stats.norm.cdf(field[i] / sigma[i])
                u_j = stats.norm.cdf(field[j] / sigma[j])
                u_i_list.append(u_i)
                u_j_list.append(u_j)

        u_i = np.array(u_i_list)
        u_j = np.array(u_j_list)

        # Test: For Gaussian copula, (Phi^{-1}(u_i), Phi^{-1}(u_j)) should
        # be bivariate Gaussian. Test using quadrant dependence.
        z_i = stats.norm.ppf(np.clip(u_i, 1e-6, 1-1e-6))
        z_j = stats.norm.ppf(np.clip(u_j, 1e-6, 1-1e-6))

        # Excess kurtosis of z_i * z_j product (should be specific value for bivar Gaussian)
        product = z_i * z_j
        kurt = stats.kurtosis(product, fisher=True)

        # Tail dependence: count concordant pairs in tails
        q = 0.9
        upper_both = np.mean((u_i > q) & (u_j > q))
        lower_both = np.mean((u_i < 1-q) & (u_j < 1-q))
        expected_tail = (1-q)**2  # independence expectation
        tail_excess = (upper_both - expected_tail)**2 + (lower_both - expected_tail)**2

        return kurt**2 + tail_excess * 1e4

    # ----- TEST 6: Integrated Polyspectra via Moments of Smoothed Fields -----
    def smoothed_moment_test(self, field, scales=[0.2, 0.4, 0.6, 0.8]):
        """
        Smooth the field at multiple angular scales, compute moments of
        the smoothed field. Higher moments of smoothed fields are sensitive
        to integrated polyspectra (bispectrum, trispectrum, etc.).
        """
        stats_list = []

        for scale in scales:
            # Gaussian smoothing kernel on sphere
            weights = np.exp(-0.5 * (self.ang_dist / scale)**2)
            weights /= weights.sum(axis=1, keepdims=True)

            smoothed = weights @ field
            s = np.std(smoothed)
            if s < 1e-10:
                continue
            normalized = smoothed / s

            # Moments 3-6
            m3 = np.mean(normalized**3)
            m4 = np.mean(normalized**4) - 3  # excess kurtosis
            m5 = np.mean(normalized**5)
            m6 = np.mean(normalized**6) - 15  # excess

            stats_list.extend([m3, m4, m5, m6])

        return np.sum(np.array(stats_list)**2)

    # ----- TEST 7: Topological (Persistent Homology inspired) -----
    def topological_test(self, field, n_thresholds=15):
        """
        Count the number of connected components and holes in excursion
        sets as a function of threshold (simplified persistence diagram).

        Non-Gaussianity affects the birth-death pattern of topological features.
        """
        hull = spatial.ConvexHull(self.coords)

        # Build adjacency list
        adj = [set() for _ in range(self.m)]
        for simplex in hull.simplices:
            for i, j in combinations(simplex, 2):
                adj[i].add(j)
                adj[j].add(i)

        sigma = np.std(field)
        thresholds = np.linspace(-2.5, 2.5, n_thresholds) * sigma

        component_counts = []
        for thresh in thresholds:
            # Count connected components of excursion set {f > thresh}
            active = field > thresh
            visited = np.zeros(self.m, dtype=bool)
            n_components = 0

            for start in range(self.m):
                if active[start] and not visited[start]:
                    # BFS
                    n_components += 1
                    queue = [start]
                    visited[start] = True
                    while queue:
                        node = queue.pop(0)
                        for neighbor in adj[node]:
                            if active[neighbor] and not visited[neighbor]:
                                visited[neighbor] = True
                                queue.append(neighbor)

            component_counts.append(n_components)

        counts = np.array(component_counts, dtype=float)

        # For Gaussian fields, the expected Betti numbers have known
        # analytical forms. Deviation is our test statistic.
        # Simplified: use variance and skewness of the count curve
        mean_c = np.mean(counts)
        if mean_c < 1e-10:
            return 0.0
        normalized_counts = counts / mean_c

        skew = stats.skew(normalized_counts)
        kurt = stats.kurtosis(normalized_counts)
        asymmetry = np.sum((counts - counts[::-1])**2) / np.sum(counts**2 + 1)

        return skew**2 + kurt**2 + asymmetry

    # ----- TEST 8: Information-theoretic (Negentropy) -----
    def negentropy_test(self, field):
        """
        Estimate negentropy J(X) = H(X_gauss) - H(X) where X_gauss has
        same covariance. Uses Hyvarinen's approximation:
        J(X) ~ sum_i [E{G_i(x)}  - E{G_i(v)}]^2
        where v ~ N(0,1) and G_i are nonlinear functions.
        """
        # Whiten
        w = self.L_inv @ field
        w = w / np.std(w)  # standardize

        # Hyvarinen's approximation with multiple nonlinearities
        def G1(x): return np.tanh(x)
        def G2(x): return x * np.exp(-x**2 / 2)
        def G3(x): return x**3

        # Expected values for N(0,1)
        # E[tanh(v)] = 0, E[v*exp(-v^2/2)] = 0, E[v^3] = 0
        # E[tanh(v)^2] ~ 0.3745 (numerical), E[(v*exp(-v^2/2))^2] ~ 0.2659

        e_g1 = np.mean(G1(w))
        e_g2 = np.mean(G2(w))
        e_g3 = np.mean(G3(w))

        # Reference values (precomputed for N(0,1))
        v = np.random.randn(100000)
        ref_g1 = np.mean(G1(v))
        ref_g2 = np.mean(G2(v))
        ref_g3 = np.mean(G3(v))

        J = (e_g1 - ref_g1)**2 + (e_g2 - ref_g2)**2 + (e_g3 - ref_g3)**2
        return J

    def run_all_tests(self, field):
        """Run all tests and return a dictionary of test statistics."""
        results = {}

        results['minkowski'] = self.minkowski_test_statistic(field)
        results['whitened_cumulants'] = self.whitened_cumulant_test(field)
        results['bispectrum'] = self.bispectrum_test(field)
        results['phase_corr'] = self.phase_correlations(field)
        results['nn_joint'] = self.nn_joint_test(field)
        results['smoothed_moments'] = self.smoothed_moment_test(field)
        results['topological'] = self.topological_test(field)
        results['negentropy'] = self.negentropy_test(field)

        return results


# Initialize test suite
print("\nInitializing test suite...")
test_suite = NonGaussianityTestSuite(coords, ang_dist, cov_matrix)

# ============================================================
# PART 6: RUN TESTS ON ENSEMBLES
# ============================================================

# Run a subset of faster tests on full ensemble, all tests on smaller subset
n_test_full = 200  # for fast tests
n_test_all = 50    # for all tests

print(f"\nRunning fast tests on {n_test_full} realizations each...")

fast_test_names = ['whitened_cumulants', 'smoothed_moments', 'negentropy', 'nn_joint']

gaussian_stats = {name: [] for name in fast_test_names}
nongaussian_stats = {name: [] for name in fast_test_names}

for i in range(n_test_full):
    if i % 50 == 0:
        print(f"  Processing realization {i}/{n_test_full}...")

    g_field = gaussian_ensemble[i]
    ng_field = nongaussian_ensemble[i]

    gaussian_stats['whitened_cumulants'].append(
        test_suite.whitened_cumulant_test(g_field))
    nongaussian_stats['whitened_cumulants'].append(
        test_suite.whitened_cumulant_test(ng_field))

    gaussian_stats['smoothed_moments'].append(
        test_suite.smoothed_moment_test(g_field))
    nongaussian_stats['smoothed_moments'].append(
        test_suite.smoothed_moment_test(ng_field))

    gaussian_stats['negentropy'].append(
        test_suite.negentropy_test(g_field))
    nongaussian_stats['negentropy'].append(
        test_suite.negentropy_test(ng_field))

    gaussian_stats['nn_joint'].append(
        test_suite.nn_joint_test(g_field))
    nongaussian_stats['nn_joint'].append(
        test_suite.nn_joint_test(ng_field))

# Run computationally heavier tests on smaller subset
print(f"\nRunning all tests on {n_test_all} realizations each...")

slow_test_names = ['bispectrum', 'phase_corr', 'topological', 'minkowski']
for name in slow_test_names:
    gaussian_stats[name] = []
    nongaussian_stats[name] = []

for i in range(n_test_all):
    if i % 10 == 0:
        print(f"  Processing realization {i}/{n_test_all}...")

    g_field = gaussian_ensemble[i]
    ng_field = nongaussian_ensemble[i]

    gaussian_stats['bispectrum'].append(test_suite.bispectrum_test(g_field))
    nongaussian_stats['bispectrum'].append(test_suite.bispectrum_test(ng_field))

    gaussian_stats['phase_corr'].append(test_suite.phase_correlations(g_field))
    nongaussian_stats['phase_corr'].append(test_suite.phase_correlations(ng_field))

    gaussian_stats['topological'].append(test_suite.topological_test(g_field))
    nongaussian_stats['topological'].append(test_suite.topological_test(ng_field))

    gaussian_stats['minkowski'].append(test_suite.minkowski_test_statistic(g_field))
    nongaussian_stats['minkowski'].append(test_suite.minkowski_test_statistic(ng_field))

# ============================================================
# PART 7: STATISTICAL ANALYSIS & DISCRIMINATION POWER
# ============================================================

print("\n" + "="*70)
print("RESULTS: NON-GAUSSIANITY DETECTION POWER")
print("="*70)

all_test_names = fast_test_names + slow_test_names

for name in all_test_names:
    g = np.array(gaussian_stats[name])
    ng = np.array(nongaussian_stats[name])

    # Remove any NaN/Inf
    g = g[np.isfinite(g)]
    ng = ng[np.isfinite(ng)]

    if len(g) < 5 or len(ng) < 5:
        print(f"\n{name}: Insufficient data")
        continue

    # KS test between distributions
    ks_stat, ks_pval = stats.ks_2samp(g, ng)

    # Mann-Whitney U test
    mw_stat, mw_pval = stats.mannwhitneyu(g, ng, alternative='two-sided')

    # Effect size (Cohen's d)
    pooled_std = np.sqrt((np.var(g) + np.var(ng)) / 2)
    cohens_d = (np.mean(ng) - np.mean(g)) / pooled_std if pooled_std > 0 else 0

    # ROC-AUC (how well can we classify Gaussian vs non-Gaussian)
    labels = np.concatenate([np.zeros(len(g)), np.ones(len(ng))])
    scores = np.concatenate([g, ng])

    # Simple AUC calculation
    thresholds = np.percentile(scores, np.linspace(0, 100, 200))
    tpr = [np.mean(ng > t) for t in thresholds]
    fpr = [np.mean(g > t) for t in thresholds]
    # Sort by fpr
    sorted_idx = np.argsort(fpr)
    fpr_sorted = np.array(fpr)[sorted_idx]
    tpr_sorted = np.array(tpr)[sorted_idx]
    auc = np.trapz(tpr_sorted, fpr_sorted)
    auc = max(auc, 1 - auc)  # ensure AUC > 0.5

    print(f"\n--- {name.upper()} ---")
    print(f"  Gaussian:     mean={np.mean(g):.6f}  std={np.std(g):.6f}")
    print(f"  Non-Gaussian: mean={np.mean(ng):.6f}  std={np.std(ng):.6f}")
    print(f"  KS test:      D={ks_stat:.4f}  p={ks_pval:.2e}")
    print(f"  Mann-Whitney:                   p={mw_pval:.2e}")
    print(f"  Cohen's d:    {cohens_d:.4f}")
    print(f"  ROC AUC:      {auc:.4f}")

    if ks_pval < 0.01:
        print(f"  *** DETECTION: Strong evidence of non-Gaussianity (p < 0.01) ***")
    elif ks_pval < 0.05:
        print(f"  ** MARGINAL: Some evidence of non-Gaussianity (p < 0.05) **")
    else:
        print(f"  ( ) No significant detection (p = {ks_pval:.3f})")

# ============================================================
# PART 8: COMBINED TEST (Fisher's method & Machine Learning-like)
# ============================================================

print("\n" + "="*70)
print("COMBINED ANALYSIS")
print("="*70)

# Use the fast tests that ran on the same realizations
n_combined = n_test_full

# Build feature matrix
def build_features(ensemble, test_suite, n_samples):
    features = np.zeros((n_samples, len(fast_test_names)))
    for i in range(n_samples):
        field = ensemble[i]
        features[i, 0] = test_suite.whitened_cumulant_test(field)
        features[i, 1] = test_suite.smoothed_moment_test(field)
        features[i, 2] = test_suite.negentropy_test(field)
        features[i, 3] = test_suite.nn_joint_test(field)
    return features

# We already computed these, reuse
g_features = np.column_stack([gaussian_stats[name] for name in fast_test_names])
ng_features = np.column_stack([nongaussian_stats[name] for name in fast_test_names])

# Normalize features
all_features = np.vstack([g_features, ng_features])
feat_mean = all_features.mean(axis=0)
feat_std = all_features.std(axis=0)
feat_std[feat_std < 1e-10] = 1

g_norm = (g_features - feat_mean) / feat_std
ng_norm = (ng_features - feat_mean) / feat_std

# Fisher Linear Discriminant
mean_g = g_norm.mean(axis=0)
mean_ng = ng_norm.mean(axis=0)
S_w = np.cov(g_norm.T) + np.cov(ng_norm.T)  # within-class scatter
try:
    w_fisher = np.linalg.solve(S_w, mean_ng - mean_g)
    w_fisher /= np.linalg.norm(w_fisher)

    proj_g = g_norm @ w_fisher
    proj_ng = ng_norm @ w_fisher

    # Combined test
    ks_combined, p_combined = stats.ks_2samp(proj_g, proj_ng)

    pooled = np.sqrt((np.var(proj_g) + np.var(proj_ng)) / 2)
    d_combined = (np.mean(proj_ng) - np.mean(proj_g)) / pooled if pooled > 0 else 0

    print(f"\nFisher Linear Discriminant (combined {len(fast_test_names)} tests):")
    print(f"  Fisher weights: {dict(zip(fast_test_names, w_fisher.round(3)))}")
    print(f"  KS test: D={ks_combined:.4f}  p={p_combined:.2e}")
    print(f"  Cohen's d: {d_combined:.4f}")

    if p_combined < 0.001:
        print(f"  *** STRONG COMBINED DETECTION ***")
except np.linalg.LinAlgError:
    print("  Fisher discriminant: singular scatter matrix")

# Hotelling's T^2 test (multivariate)
n1, n2 = len(g_norm), len(ng_norm)
S_pooled = ((n1-1)*np.cov(g_norm.T) + (n2-1)*np.cov(ng_norm.T)) / (n1+n2-2)
diff = mean_ng - mean_g
try:
    T2 = (n1*n2)/(n1+n2) * diff @ np.linalg.solve(S_pooled, diff)
    p = len(fast_test_names)
    F_stat = T2 * (n1+n2-p-1) / (p*(n1+n2-2))
    F_pval = 1 - stats.f.cdf(F_stat, p, n1+n2-p-1)

    print(f"\nHotelling's T² test:")
    print(f"  T² = {T2:.4f}")
    print(f"  F({p}, {n1+n2-p-1}) = {F_stat:.4f}")
    print(f"  p-value = {F_pval:.2e}")
except np.linalg.LinAlgError:
    print("  Hotelling's T²: singular pooled covariance")

# ============================================================
# PART 9: VISUALIZATION
# ============================================================

fig = plt.figure(figsize=(20, 24))

# 1. Example fields on sphere
ax1 = fig.add_subplot(4, 3, 1, projection='3d')
g_example = gaussian_ensemble[0]
sc = ax1.scatter(coords[:,0], coords[:,1], coords[:,2],
                c=g_example, cmap='RdBu_r', s=15,
                vmin=-3*np.std(g_example), vmax=3*np.std(g_example))
ax1.set_title('Gaussian Field', fontsize=11)
ax1.set_axis_off()

ax2 = fig.add_subplot(4, 3, 2, projection='3d')
ng_example = nongaussian_ensemble[0]
sc = ax2.scatter(coords[:,0], coords[:,1], coords[:,2],
                c=ng_example, cmap='RdBu_r', s=15,
                vmin=-3*np.std(ng_example), vmax=3*np.std(ng_example))
ax2.set_title('Non-Gaussian Field', fontsize=11)
ax2.set_axis_off()

ax3 = fig.add_subplot(4, 3, 3, projection='3d')
diff = ng_example - g_example
sc = ax3.scatter(coords[:,0], coords[:,1], coords[:,2],
                c=diff, cmap='PiYG', s=15,
                vmin=-3*np.std(diff), vmax=3*np.std(diff))
ax3.set_title('Difference (NG - G)', fontsize=11)
ax3.set_axis_off()

# 2. Test statistic distributions
for idx, name in enumerate(fast_test_names):
    ax = fig.add_subplot(4, 3, 4 + idx)
    g = np.array(gaussian_stats[name])
    ng = np.array(nongaussian_stats[name])

    g = g[np.isfinite(g)]
    ng = ng[np.isfinite(ng)]

    # Use common range
    all_vals = np.concatenate([g, ng])
    lo, hi = np.percentile(all_vals, [1, 99])
    bins = np.linspace(lo, hi, 40)

    ax.hist(g, bins=bins, alpha=0.6, density=True, label='Gaussian', color='steelblue')
    ax.hist(ng, bins=bins, alpha=0.6, density=True, label='Non-Gaussian', color='coral')
    ax.axvline(np.mean(g), color='steelblue', linestyle='--', linewidth=2)
    ax.axvline(np.mean(ng), color='coral', linestyle='--', linewidth=2)
    ax.set_title(f'{name}', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

# 3. Slow test distributions
for idx, name in enumerate(slow_test_names):
    ax = fig.add_subplot(4, 3, 8 + idx)
    g = np.array(gaussian_stats[name])
    ng = np.array(nongaussian_stats[name])

    g = g[np.isfinite(g)]
    ng = ng[np.isfinite(ng)]

    if len(g) < 5 or len(ng) < 5:
        ax.text(0.5, 0.5, 'Insufficient data', transform=ax.transAxes, ha='center')
        continue

    all_vals = np.concatenate([g, ng])
    lo, hi = np.percentile(all_vals, [2, 98])
    bins = np.linspace(lo, hi, 25)

    ax.hist(g, bins=bins, alpha=0.6, density=True, label='Gaussian', color='steelblue')
    ax.hist(ng, bins=bins, alpha=0.6, density=True, label='Non-Gaussian', color='coral')
    ax.axvline(np.mean(g), color='steelblue', linestyle='--', linewidth=2)
    ax.axvline(np.mean(ng), color='coral', linestyle='--', linewidth=2)
    ax.set_title(f'{name}', fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylabel('Density')

plt.tight_layout()
plt.savefig('nongaussianity_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved to nongaussianity_detection.png")

# ============================================================
# PART 10: POWER ANALYSIS — Detection probability vs coupling strength
# ============================================================

print("\n" + "="*70)
print("POWER ANALYSIS: Detection probability vs coupling strength")
print("="*70)

coupling_strengths = [0.02, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35]
n_power = 80
alpha_level = 0.05

power_results = {name: [] for name in ['whitened_cumulants', 'smoothed_moments']}

for eps in coupling_strengths:
    print(f"\n  Coupling strength epsilon = {eps}...")

    inj = HiddenCorrelationInjector(
        coords, ang_dist, n_groups=40,
        group_orders=[4, 5, 6], coupling_strength=eps
    )

    rng_g = np.random.RandomState(999)
    rng_ng = np.random.RandomState(999)

    g_stats = {name: [] for name in power_results}
    ng_stats = {name: [] for name in power_results}

    for i in range(n_power):
        g_field = generate_gaussian_field(L_chol, rng_g)

        # Non-Gaussian with this coupling strength
        g_field2 = generate_gaussian_field(L_chol, rng_ng)
        ng_field = inj.inject(g_field2, rng_ng)
        ng_field = inj.copula_inject(ng_field, rng_ng)

        for name in power_results:
            if name == 'whitened_cumulants':
                g_stats[name].append(test_suite.whitened_cumulant_test(g_field))
                ng_stats[name].append(test_suite.whitened_cumulant_test(ng_field))
            elif name == 'smoothed_moments':
                g_stats[name].append(test_suite.smoothed_moment_test(g_field))
                ng_stats[name].append(test_suite.smoothed_moment_test(ng_field))

    for name in power_results:
        g = np.array(g_stats[name])
        ng = np.array(ng_stats[name])

        # Power: fraction of NG realizations that exceed the 95th percentile
        # of the Gaussian distribution
        threshold = np.percentile(g, 100 * (1 - alpha_level))
        power = np.mean(ng > threshold)
        power_results[name].append(power)
        print(f"    {name}: power = {power:.3f}")

# Plot power curves
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
for name in power_results:
    ax.plot(coupling_strengths, power_results[name], 'o-', label=name, linewidth=2, markersize=8)
ax.axhline(alpha_level, color='gray', linestyle='--', label=f'Size (α={alpha_level})')
ax.axhline(0.8, color='green', linestyle=':', alpha=0.5, label='80% power')
ax.set_xlabel('Coupling Strength (ε)', fontsize=13)
ax.set_ylabel('Detection Power', fontsize=13)
ax.set_title('Power Analysis: Non-Gaussianity Detection vs Coupling Strength', fontsize=14)
ax.legend(fontsize=11)
ax.set_ylim(-0.05, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('power_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"""
System Configuration:
  - {m} degrees of freedom on a sphere (Fibonacci lattice)
  - Spatial correlations via angular power spectrum (ell_max=25)
  - Hidden {len(injector.groups)} groups with {injector.group_orders}-point correlations
  - Coupling strength: {injector.coupling_strength}

8 Independent Non-Gaussianity Tests:
  1. Minkowski Functionals (morphology of excursion sets)
  2. Whitened Cumulants (higher moments after decorrelation)
  3. Bispectrum (3-point function in angular bins)
  4. Phase Correlations (harmonic phase uniformity)
  5. Nearest-Neighbor Joint Distribution (copula departures)
  6. Smoothed Field Moments (integrated polyspectra)
  7. Topological Features (connected components vs threshold)
  8. Negentropy (information-theoretic Gaussianity measure)

Combined Analysis:
  - Fisher Linear Discriminant (optimal linear combination)
  - Hotelling's T² (multivariate hypothesis test)
""")
